# Archived: experiment to reduce dense-layer width

Compare a 64-unit Dense layer with the saved 128-unit baseline. Keep the writer split, seed, optimizer, learning rate, convolutions, batch size, and stopping rule unchanged. Train a fresh model; do not fine-tune the baseline. This is a single-seed exploratory comparison. The test set remains unused.

## 1. Imports

In [ ]:
#%load_ext autoreload
#%autoreload 2
import sys
from pathlib import Path
import matplotlib.pyplot as plt
import tensorflow as tf

keras = tf.keras
layers = keras.layers
Sequential = keras.Sequential

project_root = Path.cwd()
if project_root.name == "notebooks":
    project_root = project_root.parent

if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

from src.load_metadata import load_metadata
from src.split_dataset import split
from src.input_pipeline import build_dataset


## 2. Fixed configuration

Only the hidden Dense width changes: 128 to 64. All other settings match notebook 04.

In [ ]:
SEED = 12
BATCH_SIZE = 32
LEARNING_RATE = 0.001
FILTERS = (32, 64)
KERNEL_SIZE = 3
DENSE_UNITS = 64
MAX_EPOCHS = 10
PATIENCE = 3
INPUT_SHAPE = (63, 64, 1)

keras.utils.set_random_seed(SEED)

## 3. Data and shared labels

Pixels are already normalized by the input pipeline. Do not add another rescaling layer.

In [ ]:
metadata = load_metadata()

df_train, df_valid, df_test = split(
    metadata,
    training=0.7,
    validation=0.15,
    test=0.15,
    seed=SEED,
)

unique_char = sorted(metadata["char"].unique())
char_dic = {char: index for index, char in enumerate(unique_char)}
df_train["label"] = df_train["char"].map(char_dic)
df_valid["label"] = df_valid["char"].map(char_dic)
df_test["label"] = df_test["char"].map(char_dic)


train_ds = build_dataset(df_train, batch_size=BATCH_SIZE, training=True, seed=SEED)
val_ds = build_dataset(df_valid, batch_size=BATCH_SIZE, training=False)

## 4. Model

In [ ]:
keras.backend.clear_session()
keras.utils.set_random_seed(SEED)

model = keras.Sequential([
    layers.Input(shape=INPUT_SHAPE),
    layers.Conv2D(FILTERS[0], KERNEL_SIZE, activation="relu"),
    layers.MaxPooling2D(),
    layers.Conv2D(FILTERS[1], KERNEL_SIZE, activation="relu"),
    layers.Flatten(),
    layers.Dense(DENSE_UNITS, activation="relu"),
    layers.Dense(len(unique_char), activation="softmax"),
])

model.compile(
    optimizer=keras.optimizers.Adam(learning_rate=LEARNING_RATE),
    loss=keras.losses.SparseCategoricalCrossentropy(),
    metrics=["accuracy"],
)
model.summary()

## 5. Check a batch without updating weights

In [ ]:
for images, labels in train_ds.take(1):
    predictions = model(images, training=False)
    print("Images:", images.shape)
    print("Labels:", labels.shape)
    print("Predictions:", predictions.shape)
    assert tuple(images.shape[1:]) == INPUT_SHAPE
    assert predictions.shape == (images.shape[0], len(unique_char))

## 6. Training

Stop after three consecutive epochs without an improvement in validation loss, and restore the weights from the best epoch. Ten epochs is the maximum, not a required training duration.

In [ ]:
early_stop = keras.callbacks.EarlyStopping(
    monitor="val_loss",
    patience=PATIENCE,
    restore_best_weights=True,
)

history = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=MAX_EPOCHS,
    callbacks=[early_stop],
    shuffle=False,  # Training examples are already shuffled by the input pipeline.
)

## 7. Training and validation loss

Compare both curves. A decreasing training loss alongside an increasing validation loss suggests overfitting. The dotted line marks the epoch selected by EarlyStopping. These curves describe the training history before the best weights are restored.

In [ ]:
epochs = range(1, len(history.history["loss"]) + 1)
best_epoch = min(epochs, key=lambda epoch: history.history["val_loss"][epoch - 1])

fig, ax = plt.subplots(figsize=(7, 4))
ax.plot(epochs, history.history["loss"], label="Training")
ax.plot(epochs, history.history["val_loss"], label="Validation")
ax.axvline(best_epoch, linestyle=":", color="gray", label=f"Lowest val_loss: epoch {best_epoch}")
ax.set(xlabel="Epoch", ylabel="Cross-entropy loss", title="Dense-64 learning curves")
ax.set_xticks(list(epochs))
ax.legend()
ax.grid(alpha=0.2)
fig.tight_layout()
plt.show()

In [ ]:
models_dir = project_root / "models"
models_dir.mkdir(exist_ok=True)

model.save(models_dir / "katakana_dense64.keras")

import json

(models_dir / "katakana_dense64_history.json").write_text(
    json.dumps(history.history, indent=2), encoding="utf-8",
)
(models_dir / "katakana_dense64_classes.json").write_text(
    json.dumps(unique_char, ensure_ascii=False, indent=2), encoding="utf-8",
)

## Global comparison with the saved baseline

Compare validation accuracy, validation loss, parameter count, and total errors on the same unshuffled validation set. Both saved models use the weights selected by EarlyStopping. Specific character confusions are exploratory diagnostics, not model-selection criteria. One run cannot establish a robust improvement.

To compare without retraining, run imports, configuration, and data preparation, then this section. Skip model construction, training, plotting, and saving.

In [ ]:
import numpy as np
import pandas as pd

# This comparison can run without retraining: load both saved models.
val_ds = build_dataset(df_valid, batch_size=BATCH_SIZE, training=False)
true_labels = df_valid["label"].to_numpy()
comparison_rows = []
for name, filename in [
    ("Dense 128 (baseline)", "katakana_baseline.keras"),
    ("Dense 64", "katakana_dense64.keras"),
]:
    candidate = keras.models.load_model(project_root / "models" / filename, compile=False)
    candidate.compile(
        loss=keras.losses.SparseCategoricalCrossentropy(), metrics=["accuracy"],
    )
    metrics = candidate.evaluate(val_ds, verbose=0, return_dict=True)
    error_count = round((1 - metrics["accuracy"]) * len(df_valid))
    comparison_rows.append({
        "Model": name,
        "Parameters": candidate.count_params(),
        "Validation loss": metrics["loss"],
        "Validation accuracy": metrics["accuracy"],
        "Validation errors": error_count,
    })

comparison = pd.DataFrame(comparison_rows).set_index("Model")
display(comparison.style.format({
    "Parameters": "{:,.0f}",
    "Validation loss": "{:.4f}",
    "Validation accuracy": "{:.2%}",
}))

## Optional exploratory analysis: character confusions

Use this section to investigate error patterns after the global comparison. It does not define the criteria for choosing a model. To inspect the saved Dense-64 model without retraining, first run `model = keras.models.load_model(project_root / "models" / "katakana_dense64.keras")`.

In [ ]:
import numpy as np
import pandas as pd

val_ds = build_dataset(
    df_valid,
    batch_size=BATCH_SIZE,
    training=False,
)

probabilities = model.predict(val_ds)
predicted_labels = probabilities.argmax(axis=1)
true_labels = df_valid["label"].to_numpy()

assert len(predicted_labels) == len(true_labels)

characters = np.asarray(unique_char)

results = pd.DataFrame({
    "Actual": characters[true_labels],
    "Predicted": characters[predicted_labels],
})
results["Correct"] = results["Actual"] == results["Predicted"]

# Summary for each true character.
summary = results.groupby("Actual").agg(
    Support=("Correct", "size"),
    Recall=("Correct", "mean"),
)
summary["Errors"] = (
    summary["Support"] * (1 - summary["Recall"])
).round().astype(int)

confusions = (
    results.loc[~results["Correct"]]
    .groupby(["Actual", "Predicted"])
    .size()
    .rename("Count")
    .reset_index()
    .join(summary, on="Actual")
)

confusions["Rate among all examples"] = (
    confusions["Count"] / confusions["Support"]
)
confusions["Share of errors"] = (
    confusions["Count"] / confusions["Errors"]
)

top_confusions = (
    confusions
    .sort_values(
        ["Actual", "Count", "Predicted"],
        ascending=[True, False, True],
    )
    .groupby("Actual", sort=False)
    .head(3)
    .sort_values(
        ["Recall", "Actual", "Count"],
        ascending=[True, True, False],
    )
)

display(
    top_confusions[
        [
            "Actual",
            "Predicted",
            "Support",
            "Recall",
            "Count",
            "Rate among all examples",
            "Share of errors",
        ]
    ]
    .reset_index(drop=True)
    .style
    .format({
        "Recall": "{:.1%}",
        "Rate among all examples": "{:.1%}",
        "Share of errors": "{:.1%}",
    })
    .hide(axis="index")
)